In [ ]:
# ==============================================================================
# Module 1: Dependencies and Global Configurations
# Description: Imports required libraries, suppresses minor warnings, and 
#              establishes global constants and random seeds for reproducibility.
# ==============================================================================
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy.stats import pearsonr, gaussian_kde
from sklearn.metrics import r2_score, mean_squared_error
from pathlib import Path
from typing import List

# (Assuming hddm is available in the active environment)
import hddm

# Suppress non-critical FutureWarnings for cleaner console output during plotting
warnings.filterwarnings('ignore', category=FutureWarning)

# ==============================================================================
# Global Parameter Configurations
# ==============================================================================
GLOBAL_SEED: int = 42
PPC_SAMPLES: int = 500
CONDITION_COL: str = 'condition'

# Directory Paths Configuration
# Base directory containing the fitted .nc model files
BASE_INPUT_DIR: Path = Path("output/formal_models")  
# Base directory for exporting Posterior Predictive Checks (PPC) and related figures
BASE_OUTPUT_DIR: Path = Path("../../reference_outputs/hddm/result/formal_model") 
# Filepath for the raw, aligned trial-level dataset
RAW_CSV_PATH: Path = Path("data/xab_main_trial_level_combined.csv") 

# Target Models for Execution
MODELS_TO_RUN: List[str] = [
    "M4_v_judgment_x_pair", 
    # "M6_v_interaction_a_train", 
    # "M8_av_judgment_x_pair"
]

def set_global_seed(seed: int = 42) -> None:
    """
    Sets the global random seed across standard Python, NumPy, and system 
    environments to guarantee strict reproducibility of stochastic processes 
    (e.g., MCMC sampling, KDE estimations, and posterior simulations).

    Args:
        seed (int): The integer value to use as the random seed. Defaults to 42.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"[INFO] Global random seed established: {seed} (Reproducibility enforced)")

# Initialize the global seed environment prior to any computations
set_global_seed(GLOBAL_SEED)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from pathlib import Path
from typing import Any

# ==============================================================================
# Module: Publication-Ready 2xN Collapsed PPC Distribution Plot
# Description: Generates high-fidelity, 4:3 aspect ratio Posterior Predictive 
#              Check (PPC) density plots with optimized two-line academic labels.
# ==============================================================================

def plot_hddm_ppc_collapsed_conditions(
    model: Any, 
    ppc_data: pd.DataFrame, 
    condition_col: str, 
    num_samples_to_plot: int = 50, 
    model_name: str = "my_model"
) -> None:
    """
    Plots the observed reaction time (RT) distributions against posterior 
    predictive samples to evaluate model fit across different conditions.

    Args:
        model (Any): The fitted HDDM model object.
        ppc_data (pd.DataFrame): The generated posterior predictive check data.
        condition_col (str): The column name representing the experimental condition.
        num_samples_to_plot (int): Number of posterior predictive traces to overlay.
        model_name (str): Identifier for the model, used for saving outputs.
    """
    print(f"[INFO] Generating publication-ready 2xN collapsed PPC density plots for {model_name}...")
    
    model_data = model.data
    
    # Check if conditions need mapping (assumes map_conditions_to_ppc is defined elsewhere)
    if condition_col not in ppc_data.columns:
        ppc_data = map_conditions_to_ppc(model, ppc_data, condition_col)
        
    conditions = model_data[condition_col].unique()
    
    # Establish academic color palette
    color_obs = "#8DA0CB"  # Distinct blue for observed data
    color_sim = "#FC8D62"  # Contrasting orange for simulated data
    
    sns.set_theme(style="white", font_scale=1.5)
    cols = min(2, len(conditions))
    rows = int(np.ceil(len(conditions) / cols))
    
    # Enforce standard 4:3 academic aspect ratio (16x12 figure size)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12)) 
    axes = np.array([axes]).flatten() if len(conditions) == 1 else axes.flatten()
    
    # Randomly select a subset of traces for visualization to avoid overplotting
    plot_samples = np.random.choice(
        ppc_data.index.get_level_values('sample').unique(), 
        size=num_samples_to_plot, 
        replace=False
    )
    
    for i, cond in enumerate(conditions):
        ax = axes[i]
        obs_cond = model_data[model_data[condition_col] == cond]
        ppc_cond = ppc_data[ppc_data[condition_col] == cond]
        
        if len(obs_cond) == 0: 
            continue
            
        ax.grid(False)
        
        # Thicken spines for a bold, publication-quality appearance
        for spine in ['bottom', 'left']:
            ax.spines[spine].set_linewidth(3.0) 
            ax.spines[spine].set_color('black')
        
        # Optimize tick parameters for readability
        ax.tick_params(axis='both', which='major', labelsize=26, width=3.0, length=10, colors='black')
        
        # Standardize Y-axis tick intervals
        ax.set_yticks([0, 0.5, 1.0])
        ax.set_yticklabels(['0', '0.5', '1.0'])
        
        # Extract and bound reaction time data
        obs_rt = obs_cond['rt'].abs().dropna()
        max_rt = obs_rt.max() * 1.15
        x_pos = np.linspace(0.01, max_rt, 300)
        
        # Plot observed data as an underlying histogram
        ax.hist(obs_rt, bins=30, density=True, color=color_obs, edgecolor='white', alpha=0.85, zorder=2)
        
        # Overlay simulated posterior predictive KDE traces
        for s in plot_samples:
            sim_df = ppc_cond.xs(s, level='sample', drop_level=False)
            if len(sim_df) == 0: 
                continue
            sim_rt = sim_df['rt'].abs().dropna()
            
            if len(sim_rt) > 2: 
                ax.plot(x_pos, gaussian_kde(sim_rt)(x_pos), color=color_sim, alpha=0.15, zorder=3)

        # ==============================================================================
        # Core Formatting: Replace underscores with newlines to form compact, 
        # two-line categorical labels inside the plot space.
        # ==============================================================================
        clean_cond = str(cond).title().replace('_', '\n')
        
        # Adjust linespacing and weight to ensure harmonious alignment
        ax.text(
            0.95, 0.95, clean_cond, transform=ax.transAxes, 
            fontsize=24, fontweight='bold', va='top', ha='right', 
            linespacing=1.3, zorder=5
        )
        
        # Remove individual sub-axis labels in favor of global figure labels
        ax.set_xlabel("") 
        ax.set_ylabel("")
            
        ax.set_xlim(0, max_rt)
        
    # Delete unused subplot axes dynamically
    for j in range(i + 1, len(axes)): 
        fig.delaxes(axes[j])
    
    sns.despine()
    
    # Establish centralized global axis titles
    fig.supxlabel("Reaction Time (s)", fontsize=36, fontweight='bold', color='black', x=0.5, y=0.03)
    fig.supylabel("Density", fontsize=36, fontweight='bold', color='black', x=0.02, y=0.5)

    # Construct custom global legend
    obs_patch = mpatches.Patch(color=color_obs, label='Observed Data')
    sim_line = mlines.Line2D([], [], color=color_sim, lw=5, label='Predicted Data')
    
    fig.legend(
        handles=[obs_patch, sim_line], loc='upper center', 
        bbox_to_anchor=(0.5, 1.05), ncol=2, frameon=False, 
        prop={'size': 32, 'weight': 'bold'}
    )
    
    plt.tight_layout(rect=[0.05, 0.05, 0.98, 0.95])
    
    # Dynamically resolve save directory path with fallback mechanism
    try:
        # Assuming BASE_OUTPUT_DIR is defined in the global scope from previous module
        save_dir = Path(BASE_OUTPUT_DIR) / model_name
    except NameError:
        save_dir = Path("output/result/formal_models") / model_name
        
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"ppc_collapsed_conditions_{condition_col}.png"
    
    plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"[SUCCESS] Optimized Collapsed 2xN KDE plots exported to: {save_path}")

In [ ]:
# ==============================================================================
# Module 2: Core Functions for Posterior Predictive Checks (PPC)
# Description: Generates, maps, and evaluates posterior predictive samples. 
#              Includes advanced alignment mechanisms to map simulated data back 
#              to empirical trial-level indices.
# ==============================================================================

import random
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from pathlib import Path
from typing import Any, Dict, Optional, Union

def manual_parents_to_joint_posterior_sample(
    bottom_node: Any, 
    stacked_posterior: az.InferenceData, 
    sample_idx: int
) -> None:
    """
    Manually injects posterior samples into the PyMC graph nodes to reconstruct 
    the joint posterior state for a given sampling iteration.
    """
    missing_deterministic = []
    for parent in bottom_node.extended_parents:
        if not isinstance(parent, pm.Node):  
            continue
            
        param_name = parent.__name__
        clean_name = param_name.replace("('", "_").replace("')", "").replace(".", "_")
        
        target_name = None
        if param_name in stacked_posterior:
            target_name = param_name
        elif clean_name in stacked_posterior:
            target_name = clean_name
            
        if target_name is not None:
            val = stacked_posterior[target_name].isel(draw_idx=sample_idx).values
            parent.value = val.item() if val.size == 1 else val
        else:
            if isinstance(parent, pm.Stochastic):
                raise RuntimeError(
                    f"[ERROR] Fatal: Stochastic parameter node '{param_name}' "
                    f"could not be mapped within the ArviZ trace!"
                )
            else:
                missing_deterministic.append(param_name)
                
    if missing_deterministic and not getattr(bottom_node, '_warned_missing', False):
        bottom_node._warned_missing = True


def manual_post_pred_gen_from_nc(
    model: Any, 
    idata: az.InferenceData, 
    samples: int = 500, 
    append_data: bool = True, 
    seed: int = 42
) -> pd.DataFrame:
    """
    Generates posterior predictive check (PPC) data directly from an ArviZ 
    InferenceData object, bypassing default PyMC generator limitations.
    """
    results = {}
    np.random.seed(seed)
    
    stacked_posterior = idata.posterior.stack(draw_idx=("chain", "draw"))
    total_samples = stacked_posterior.sizes["draw_idx"]
    random_indices = np.random.choice(
        total_samples, size=samples, replace=(samples > total_samples)
    )
    
    for name, obs in model.iter_observeds():
        node = obs["node"]
        try:
            data = model.data.loc[node.value.index].copy()
        except AttributeError:
            data = None
            
        datasets = []
        for i in range(samples):
            idx = random_indices[i]
            manual_parents_to_joint_posterior_sample(node, stacked_posterior, idx)
            sampled_data = node.random()
            
            if not isinstance(sampled_data, pd.DataFrame):
                sampled_data = pd.DataFrame(sampled_data)
                
            if data is not None:
                sampled_data.index = data.index 
                if append_data:
                    sampled_data = sampled_data.rename(
                        columns={'rt': 'rt_sim', 'response': 'response_sim'}
                    )
                    sampled_data = sampled_data.join(data)
            datasets.append(sampled_data)
            
        results[name] = pd.concat(
            datasets, keys=list(range(samples)), names=['sample', 'trial_idx']
        )
        
    final_ppc_data = pd.concat(results, names=['node'])
    
    if append_data and 'rt_sim' in final_ppc_data.columns:
        final_ppc_data = final_ppc_data.rename(columns={
            'rt': 'rt_obs', 'response': 'response_obs',
            'rt_sim': 'rt', 'response_sim': 'response'
        })
        
    return final_ppc_data


def get_or_generate_ppc_data(
    model: Any, 
    idata: az.InferenceData, 
    condition_col: str, 
    samples: int = 500, 
    seed: int = 42, 
    model_name: str = "my_model", 
    force_rebuild: bool = False
) -> pd.DataFrame:
    """
    Retrieves cached PPC data from disk, or generates and saves it if unavailable.
    """
    # Assuming BASE_OUTPUT_DIR is defined globally as Path("output/result/formal_model")
    save_dir = Path(BASE_OUTPUT_DIR) / model_name
    save_dir.mkdir(parents=True, exist_ok=True) 
    file_path = save_dir / f"ppc_data_{condition_col}.pkl"
    
    if file_path.exists() and not force_rebuild:
        print(f"[INFO] Loading cached PPC data from {file_path}...")
        ppc_data = pd.read_pickle(file_path)
    else:
        print(f"[INFO] Generating new PPC data for {model_name}...")
        ppc_data = manual_post_pred_gen_from_nc(
            model, idata, samples=samples, append_data=True, seed=seed
        )
        ppc_data.to_pickle(file_path)
        
    return ppc_data


def map_conditions_to_ppc(
    model: Any, 
    ppc_data: pd.DataFrame, 
    condition_col: str
) -> pd.DataFrame:
    """
    Aligns and appends categorical condition labels from the original empirical 
    dataset to the generated PPC traces based on internal indices.
    """
    if condition_col in ppc_data.columns: 
        return ppc_data
        
    model_data = model.data
    ppc_data_mapped = ppc_data.copy()
    cond_dict = model_data[condition_col].to_dict()
    
    try:
        mapped = ppc_data_mapped.index.get_level_values(2).map(cond_dict)
        if not mapped.isnull().all():
            ppc_data_mapped[condition_col] = mapped
            return ppc_data_mapped
    except Exception: 
        pass

    ppc_reset = ppc_data_mapped.reset_index()
    cols = list(ppc_reset.columns)
    if len(cols) > 2: 
        cols[2] = 'trial_idx'
    ppc_reset.columns = cols
    
    cond_list = []
    for name, obs in model.iter_observeds():
        node_data = model_data.loc[obs['node'].value.index]
        cond_values = node_data[condition_col].values
        tmp = pd.DataFrame({
            'node': name, 
            'trial_idx': range(len(cond_values)), 
            condition_col: cond_values
        })
        cond_list.append(tmp)
        
    cond_map = pd.concat(cond_list, ignore_index=True)
    ppc_merged = pd.merge(ppc_reset, cond_map, on=['node', 'trial_idx'], how='left')
    return ppc_merged.set_index(['node', 'sample', 'trial_idx'])


def save_ppc_stats_table(
    model: Any, 
    ppc_data: pd.DataFrame, 
    condition_col: str, 
    model_name: str = "my_model"
) -> pd.DataFrame:
    """
    Computes statistical goodness-of-fit metrics (Bayesian P-values, 95% HDIs) 
    comparing observed accuracy and RT percentiles against posterior predictions.
    """
    model_data = model.data
    ppc_data = map_conditions_to_ppc(model, ppc_data, condition_col)
    conditions = model_data[condition_col].unique()
    results = []
    
    for cond in conditions:
        obs_cond_data = model_data[model_data[condition_col] == cond]
        ppc_cond_data = ppc_data[ppc_data[condition_col] == cond]
        
        # Accuracy Metrics
        obs_acc = obs_cond_data['response'].mean()
        sim_accs = np.array([
            sim_df['response'].mean() for s, sim_df in ppc_cond_data.groupby(level='sample')
        ])
        sim_acc_hdi = az.hdi(sim_accs, hdi_prob=0.95)
        p_val_acc = np.mean(sim_accs >= obs_acc)
        
        results.append({
            'Condition': cond, 
            'Metric': 'Accuracy', 
            'Observed': round(obs_acc, 4),
            'Predicted_Mean': round(np.mean(sim_accs), 4), 
            '95%_HDI_Lower': round(sim_acc_hdi[0], 4),
            '95%_HDI_Upper': round(sim_acc_hdi[1], 4), 
            'Bayes_P(2-sided)': round(2 * min(p_val_acc, 1 - p_val_acc), 4)
        })
        
        # RT Percentile Metrics
        for q in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]:
            obs_q = obs_cond_data[obs_cond_data['response'] == 1]['rt'].quantile(q)
            sim_qs = np.array([
                sim_df[sim_df['response'] == 1]['rt'].quantile(q) 
                for s, sim_df in ppc_cond_data.groupby(level='sample') 
                if not sim_df[sim_df['response'] == 1].empty
            ])
            
            if len(sim_qs) > 0:
                sim_q_hdi = az.hdi(sim_qs, hdi_prob=0.95)
                p_val_rt = np.mean(sim_qs >= obs_q)
                results.append({
                    'Condition': cond, 
                    'Metric': f'RT_{int(q*100)}th_Percentile', 
                    'Observed': round(obs_q, 4),
                    'Predicted_Mean': round(np.mean(sim_qs), 4), 
                    '95%_HDI_Lower': round(sim_q_hdi[0], 4),
                    '95%_HDI_Upper': round(sim_q_hdi[1], 4), 
                    'Bayes_P(2-sided)': round(2 * min(p_val_rt, 1 - p_val_rt), 4)
                })

    df_results = pd.DataFrame(results)
    save_dir = Path(BASE_OUTPUT_DIR) / model_name
    save_dir.mkdir(parents=True, exist_ok=True)
    file_path = save_dir / f"ppc_stats_table_{condition_col}.csv"
    
    df_results.to_csv(file_path, index=False, encoding='utf-8-sig')
    return df_results


def export_aligned_ppc_data(
    model: Any, 
    ppc_df: pd.DataFrame, 
    save_path: Union[str, Path], 
    export_mode: str = "aggregate"
) -> pd.DataFrame:
    """
    Exports aligned simulated data using empirical trial indices.
    
    Export Modes:
    - 'aggregate'     : Computes the mean across all 500 simulations per trial.
    - 'single_random' : Extracts raw trial data from one random simulation iteration.
    - 'all_samples'   : Expands all simulation traces (generates a very large dataset).
    """
    print(f"[INFO] Aligning simulated data utilizing underlying empirical indices (Mode: {export_mode})...")
    
    save_path = Path(save_path)
    ppc_flat = ppc_df.reset_index()
    
    real_data = model.data.copy()
    real_data['trial_idx'] = real_data.index 
    
    # Strip negative signs from empirical data (commonly used in HDDM for incorrect trials)
    real_data['rt'] = real_data['rt'].abs()
    
    cols_to_keep = ['trial_idx', 'subj_idx', 'judgment_condition', 'pair_type', 'xab_pairs', 'response', 'rt']
    keep_cols = [c for c in cols_to_keep if c in real_data.columns]
    condition_df = real_data[keep_cols].rename(columns={
        'response': 'Observed_Acc', 
        'rt': 'Observed_RT'
    })

    sim_rt_col = 'rt_sim' if 'rt_sim' in ppc_flat.columns else ('rt_sampled' if 'rt_sampled' in ppc_flat.columns else 'rt')
    sim_resp_col = 'response_sim' if 'response_sim' in ppc_flat.columns else ('response_sampled' if 'response_sampled' in ppc_flat.columns else 'response')
    
    # Ensure simulated absolute RT is captured
    ppc_flat['rt_abs'] = ppc_flat[sim_rt_col].abs()

    # ==========================================================================
    # Mode 1: Aggregate Mode (Mean across simulations)
    # ==========================================================================
    if export_mode == "aggregate":
        print("[INFO] [Aggregate Mode] Extracting mean values across all simulations per trial...")
        acc_agg = ppc_flat.groupby('trial_idx')[sim_resp_col].mean().reset_index(name='Simulated_Acc_Mean')
        rt_all_agg = ppc_flat.groupby('trial_idx')['rt_abs'].mean().reset_index(name='Simulated_RT_Mean_All')
        
        correct_rt = ppc_flat[ppc_flat[sim_resp_col] == 1]
        rt_correct_agg = correct_rt.groupby('trial_idx')['rt_abs'].mean().reset_index(name='Simulated_RT_Mean_Correct')
        
        ppc_final = pd.merge(acc_agg, rt_all_agg, on='trial_idx', how='left')
        ppc_final = pd.merge(ppc_final, rt_correct_agg, on='trial_idx', how='left')

    # ==========================================================================
    # Mode 2: Single Random Mode (Extract one full simulation)
    # ==========================================================================
    elif export_mode == "single_random":
        available_samples = ppc_flat['sample'].unique()
        chosen_sample = random.choice(available_samples)
        print(f"[INFO] [Single Random Mode] Extracting raw trial data from simulation iteration: {chosen_sample}...")
        
        single_ppc = ppc_flat[ppc_flat['sample'] == chosen_sample].copy()
        
        single_ppc.rename(columns={
            'rt_abs': 'Simulated_RT_Mean_All', 
            sim_resp_col: 'Simulated_Acc_Mean'
        }, inplace=True)
        
        single_ppc['Simulated_RT_Mean_Correct'] = np.where(
            single_ppc['Simulated_Acc_Mean'] == 1, 
            single_ppc['Simulated_RT_Mean_All'], 
            np.nan
        )
        ppc_final = single_ppc[['trial_idx', 'Simulated_Acc_Mean', 'Simulated_RT_Mean_All', 'Simulated_RT_Mean_Correct']]

    # ==========================================================================
    # Mode 3: All Samples Mode (Flatten full dataset)
    # ==========================================================================
    elif export_mode == "all_samples":
        print("[INFO] [All Samples Mode] Expanding complete simulation dataset (Note: This will generate a large file)...")
        ppc_final = ppc_flat[['trial_idx', 'sample', 'rt_abs', sim_resp_col]].copy()
        
        ppc_final.rename(columns={
            'rt_abs': 'Simulated_RT_All', 
            sim_resp_col: 'Simulated_Acc', 
            'sample': 'sample_id'
        }, inplace=True)
        
    else:
        raise ValueError("[ERROR] Invalid export_mode parameter! Use 'aggregate', 'single_random', or 'all_samples'.")

    # ==========================================================================
    # Data Stitching and Export
    # ==========================================================================
    master_ppc = pd.merge(ppc_final, condition_df, on='trial_idx', how='left')
    
    front_cols = ['sample_id', 'trial_idx', 'subj_idx', 'judgment_condition', 'pair_type', 'xab_pairs']
    front_cols = [c for c in front_cols if c in master_ppc.columns]
    back_cols = [c for c in master_ppc.columns if c not in front_cols]
    master_ppc = master_ppc[front_cols + back_cols]
    
    save_path.parent.mkdir(parents=True, exist_ok=True)
    master_ppc.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print(f"[SUCCESS] Data successfully exported to: {save_path} (Total rows: {len(master_ppc):,})")
    
    return master_ppc

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy import stats
from scipy.stats import gaussian_kde
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union

# ==============================================================================
# Module 3: Effect Size Computation and Statistical Inference
# Description: Calculates empirical effect sizes and performs Bayesian 2x2 Mixed 
#              ANOVAs using Posterior Predictive Check (PPC) traces.
# ==============================================================================

def compute_real_effects(model_data: pd.DataFrame) -> Dict[str, float]:
    """
    Computes empirical main effects, interactions, and simple effects from the 
    observed dataset using predefined condition contrasts and partial eta squared.

    Args:
        model_data (pd.DataFrame): The empirical dataset used to fit the model.

    Returns:
        Dict[str, float]: A dictionary mapping effect names to their calculated values.
    """
    df = model_data.copy()
    df['rt_abs'] = df['rt'].abs()
    
    # Aggregate to Subject-Level
    acc_subj = df.groupby(['subj_idx', 'judgment_condition', 'pair_type'])['response'].mean().reset_index()
    rt_subj = df[df['response'] == 1].groupby(['subj_idx', 'judgment_condition', 'pair_type'])['rt_abs'].mean().reset_index()
    
    # Contrast weights defined in strict lowercase to ensure alignment with formatted data
    weights = {
        'judgment_effect': {'categorical_between': 0.5, 'categorical_within': 0.5, 'continuous_between': -0.5, 'continuous_within': -0.5},
        'pair_effect': {'categorical_between': 0.5, 'categorical_within': -0.5, 'continuous_between': 0.5, 'continuous_within': -0.5},
        'interaction': {'categorical_between': 1, 'categorical_within': -1, 'continuous_between': -1, 'continuous_within': 1}
    }
    
    real_effects = {}
    
    for metric, subj_df, val_col in [('acc', acc_subj, 'response'), ('rt', rt_subj, 'rt_abs')]:
        stats_df = subj_df.groupby(['judgment_condition', 'pair_type'])[val_col].agg(['mean', 'var', 'count']).reset_index()
        
        # Ensure condition concatenation results in strictly lowercase strings
        stats_df['cond'] = stats_df['judgment_condition'].astype(str).str.lower() + "_" + stats_df['pair_type'].astype(str).str.lower()
        
        M = stats_df.set_index('cond')['mean'].to_dict()
        V = stats_df.set_index('cond')['var'].fillna(0).to_dict()
        N = stats_df.set_index('cond')['count'].to_dict()
        
        DF_E = sum(n - 1 for n in N.values() if pd.notna(n))
        MSE = sum((N[k] - 1) * V[k] for k in N if pd.notna(N[k])) / DF_E if DF_E > 0 else np.nan
        
        for eff_name, w in weights.items():
            # Absolute Difference
            L = sum(w[k] * M.get(k, 0) for k in w)
            real_effects[f"{metric}_{eff_name}"] = L
            
            # Partial Eta Squared (Effect Size)
            if pd.notna(MSE) and MSE > 0:
                SE = np.sqrt(MSE * sum((w[k]**2) / N[k] for k in w if k in N))
                t = L / SE
                real_effects[f"{metric}_{eff_name}_eta2"] = (t**2) / (t**2 + DF_E)
            else:
                real_effects[f"{metric}_{eff_name}_eta2"] = np.nan
                
        # Simple Effects Computations
        real_effects[f"{metric}_simple_judgment_between"] = M.get('categorical_between', 0) - M.get('continuous_between', 0)
        real_effects[f"{metric}_simple_judgment_within"] = M.get('categorical_within', 0) - M.get('continuous_within', 0)
        real_effects[f"{metric}_simple_pair_categorical"] = M.get('categorical_between', 0) - M.get('categorical_within', 0)
        real_effects[f"{metric}_simple_pair_continuous"] = M.get('continuous_between', 0) - M.get('continuous_within', 0)
        
    return real_effects


def compute_ppc_effects(
    model: Any, 
    ppc_data: pd.DataFrame, 
    condition_cols: List[str] = ['judgment_condition', 'pair_type']
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Executes a Mixed ANOVA across posterior predictive samples to generate 
    simulated distributions for interaction effects, main effects, and p-values.
    """
    print("[INFO] Computing 2x2 Mixed ANOVA based on posterior predictive samples...")
    ppc_flat = ppc_data.reset_index()
    
    sim_rt_col = 'rt_sim' if 'rt_sim' in ppc_flat.columns else ('rt_sampled' if 'rt_sampled' in ppc_flat.columns else 'rt')
    sim_resp_col = 'response_sim' if 'response_sim' in ppc_flat.columns else ('response_sampled' if 'response_sampled' in ppc_flat.columns else 'response')
    ppc_flat['rt_abs'] = ppc_flat[sim_rt_col].abs()

    # Isolate Accuracy and RT datasets (RT filtered for correct responses only)
    acc_df = ppc_flat.copy()
    rt_df = ppc_flat[ppc_flat[sim_resp_col] == 1].copy() 
    
    # Aggregate to subject-level means per simulation sample
    acc_subj = acc_df.groupby(['sample', 'subj_idx'] + condition_cols)[sim_resp_col].mean().reset_index()
    rt_subj = rt_df.groupby(['sample', 'subj_idx'] + condition_cols)['rt_abs'].mean().reset_index()
    
    effects_list = []
    unique_samples = ppc_flat['sample'].unique()
    
    for s in unique_samples:
        sample_res = {'sample': s}
        
        for metric, m_df, target_col in [('acc', acc_subj[acc_subj['sample'] == s], sim_resp_col), 
                                         ('rt', rt_subj[rt_subj['sample'] == s], 'rt_abs')]:
            
            col_between = condition_cols[0] 
            col_within = condition_cols[1]  
            
            # 1. Pivot pair_type into 'between' and 'within' columns for difference scores
            pivot = m_df.pivot_table(
                index=['subj_idx', col_between], 
                columns=col_within, 
                values=target_col
            ).reset_index()
            
            # Standardize column nomenclature to lowercase
            pivot.columns = [str(c).lower() for c in pivot.columns]
            pivot[col_between] = pivot[col_between].astype(str).str.lower()
            
            # 2. Compute the core of the ANOVA: Within-subject difference scores
            if 'between' in pivot.columns and 'within' in pivot.columns:
                pivot['pair_diff'] = pivot['between'] - pivot['within']
            else:
                continue  # Skip if extreme sampling resulted in missing conditions
                
            # Isolate the two judgment groups
            cat_group = pivot[pivot[col_between] == 'categorical']
            con_group = pivot[pivot[col_between] == 'continuous']
            
            cat_diff = cat_group['pair_diff'].dropna()
            con_diff = con_group['pair_diff'].dropna()
            
            # ==================================================================
            # Core Inference: Calculate ANOVA Interaction P-value via independent t-test
            # ==================================================================
            if len(cat_diff) > 1 and len(con_diff) > 1:
                t_int, p_int = stats.ttest_ind(cat_diff, con_diff, equal_var=True)
                df_int = len(cat_diff) + len(con_diff) - 2
                eta2_p = (t_int**2) / (t_int**2 + df_int)  # Partial Eta Squared
                
                sample_res[f"{metric}_interaction_pval"] = p_int
                sample_res[f"{metric}_interaction_eta2"] = eta2_p
                # Mean Effect Size Output
                sample_res[f"{metric}_interaction"] = cat_diff.mean() - con_diff.mean()
            else:
                sample_res[f"{metric}_interaction_pval"] = np.nan
                sample_res[f"{metric}_interaction_eta2"] = np.nan
                sample_res[f"{metric}_interaction"] = np.nan

        effects_list.append(sample_res)
        
    effects_df = pd.DataFrame(effects_list)
    
    # ==========================================================================
    # Statistical Summary Reporting
    # ==========================================================================
    print("\n" + "="*50)
    for metric in ['acc', 'rt']:
        pval_col = f"{metric}_interaction_pval"
        if pval_col in effects_df.columns:
            valid_pvals = effects_df[pval_col].dropna()
            total_valid = len(valid_pvals)
            
            if total_valid > 0:
                sig_count = (valid_pvals < 0.05).sum()
                sig_ratio = sig_count / total_valid
                print(f"[INFO] {metric.upper()} Interaction ANOVA Statistics (Valid samples: {total_valid}):")
                print(f"       - Significant interactions (p < 0.05): {sig_count}")
                print(f"       - Significance ratio: {sig_ratio:.2%}")
    print("="*50 + "\n")
    
    # Generate High-Density Interval Summaries
    summary_records = []
    summary_cols = [c for c in effects_df.columns if c != 'sample' and not c.endswith('_pval')]
    
    for col in summary_cols:
        valid_data = effects_df[col].dropna().values
        if len(valid_data) > 1:
            hdi = az.hdi(valid_data, hdi_prob=0.95)
            summary_records.append({
                'Effect': col, 'Mean': np.mean(valid_data),
                '95%_HDI_Lower': hdi[0], '95%_HDI_Upper': hdi[1], 'P(>0)': np.mean(valid_data > 0)
            })
    summary_df = pd.DataFrame(summary_records)
    
    return effects_df, summary_df


# ==============================================================================
# Module 4: High-Fidelity Effect Distribution Plotting
# ==============================================================================

def plot_ppc_effects(
    samples_df: pd.DataFrame, 
    summary_df: pd.DataFrame, 
    real_effects: Dict[str, float], 
    metric: str = 'acc', 
    output_dir: Union[str, Path] = 'output/result/formal_model'
) -> None:
    """
    Generates a 3x3 comprehensive grid of normalized KDE plots comparing 
    posterior predictive effect sizes against observed empirical effects.
    """
    print(f"[INFO] Rendering 3x3 panoramic effect distributions for {metric.upper()}...")
    output_dir = Path(output_dir)
    
    effect_cols = [c for c in samples_df.columns if c.startswith(metric) and not c.endswith('_eta2')]
    
    n_cols = 3
    n_rows = int(np.ceil(len(effect_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes = np.array([axes]).flatten() if len(effect_cols) == 1 else axes.flatten()
    
    sns.set_theme(style="white", font_scale=1.2)
    
    for i, effect_name in enumerate(effect_cols):
        ax = axes[i]
        
        sim_data = samples_df[effect_name].dropna().values
        real_val = real_effects.get(effect_name, np.nan)
        
        hdi_info = summary_df[summary_df['Effect'] == effect_name]
        if not hdi_info.empty:
            hdi_low = hdi_info.iloc[0]['95%_HDI_Lower']
            hdi_high = hdi_info.iloc[0]['95%_HDI_Upper']
        else:
            hdi_low, hdi_high = np.nan, np.nan
            
        if metric == 'acc':
            sim_data = sim_data * 100
            if not np.isnan(real_val): real_val *= 100
            if not np.isnan(hdi_low): hdi_low *= 100
            if not np.isnan(hdi_high): hdi_high *= 100
        
        if len(sim_data) > 1 and np.var(sim_data) > 1e-8:
            kde = gaussian_kde(sim_data)
            x_min, x_max = sim_data.min(), sim_data.max()
            padding = np.std(sim_data) if np.std(sim_data) > 0 else 0.1
            x = np.linspace(x_min - padding, x_max + padding, 500)
            
            y = kde(x)
            # Normalize density to peak at 1.0
            y = y / np.max(y)
            
            ax.plot(x, y, color='#4A708B', linewidth=2)
            
            if not np.isnan(hdi_low) and not np.isnan(hdi_high):
                mask = (x >= hdi_low) & (x <= hdi_high)
                ax.fill_between(x[mask], 0, y[mask], color='#8C96C6', alpha=0.6)
                ax.fill_between(x[~mask], 0, y[~mask], color='#B3CDE3', alpha=0.3)
            else:
                ax.fill_between(x, 0, y, color='#B3CDE3', alpha=0.6)
                
        # Lock Y-axis scale and tick marks
        ax.set_ylim(0, 1.1)
        ax.set_yticks([0, 0.5, 1.0])
        ax.set_yticklabels(['0', '0.5', '1.0'])
            
        if not np.isnan(real_val):
            ax.axvline(real_val, color='#E41A1C', linestyle='-', linewidth=3, zorder=5)
        
        clean_title = effect_name.replace(metric + '_', '').replace('_', ' ').title()
        ax.set_title(clean_title, fontweight='bold', fontsize=16)
        
        x_label_str = "Effect Size (%)" if metric == 'acc' else "Effect Size (s)"
        ax.set_xlabel(x_label_str)
        ax.set_ylabel('Density') 
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
    for j in range(len(effect_cols), len(axes)):
        fig.delaxes(axes[j])
        
    hdi_patch = mpatches.Patch(color='#8C96C6', alpha=0.6, label='95% Model HDI')
    sim_patch = mpatches.Patch(color='#B3CDE3', alpha=0.3, label='Model Posterior')
    obs_line = mlines.Line2D([], [], color='#E41A1C', linewidth=3, label='Observed Real Effect')
    
    fig.legend(
        handles=[obs_line, hdi_patch, sim_patch], 
        loc='upper center', bbox_to_anchor=(0.5, 1.05), 
        ncol=3, frameon=False, prop={'size': 14, 'weight': 'bold'}
    )
    
    plt.tight_layout()
    output_dir.mkdir(parents=True, exist_ok=True)
    metric_label = "Accuracy" if metric == 'acc' else "Reaction_Time"
    save_path = output_dir / f"PPC_Effects_Comparison_{metric_label}.png"
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"[SUCCESS] Panoramic effect grid ({metric_label}) saved to: {save_path}")


def plot_publication_interaction(
    samples_csv: Union[str, Path], 
    summary_csv: Union[str, Path], 
    real_interaction_val: float = None, 
    metric: str = 'acc', 
    is_eta2: bool = False, 
    output_dir: Union[str, Path] = 'output/result/formal_model'
) -> None:
    """
    Generates a highly refined, publication-ready singular distribution plot 
    for evaluating specific interaction effect sizes or Partial Eta Squared.
    """
    output_dir = Path(output_dir)
    effect_type = "interaction_eta2" if is_eta2 else "interaction"
    effect_name = f"{metric}_{effect_type}"
    
    print(f"[INFO] Rendering publication-ready posterior distribution for {effect_name.upper()}...")
    
    samples_df = pd.read_csv(samples_csv)
    summary_df = pd.read_csv(summary_csv)
        
    sim_data = samples_df[effect_name].dropna().values
    stats = summary_df[summary_df['Effect'] == effect_name].iloc[0]
    hdi_low, hdi_high, mean_val, p_val = stats['95%_HDI_Lower'], stats['95%_HDI_Upper'], stats['Mean'], stats['P(>0)']

    format_pct = (metric == 'acc' and not is_eta2)
    if format_pct:
        sim_data = sim_data * 100
        hdi_low *= 100
        hdi_high *= 100
        mean_val *= 100
        if real_interaction_val is not None:
            real_interaction_val *= 100

    sns.set_theme(style="white", font_scale=1.5)
    fig, ax = plt.subplots(figsize=(11, 7))
    
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_linewidth(3.0)
        ax.spines[spine].set_color('black')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=22, width=3.0, length=10, colors='black')
    
    kde = gaussian_kde(sim_data)
    x = np.linspace(sim_data.min() - np.std(sim_data)*1.5, sim_data.max() + np.std(sim_data)*1.5, 1000) 
    
    y = kde(x)
    # Normalize density peak to 1.0
    y = y / np.max(y)
    
    # Lock Y-axis scale and tick marks
    ax.set_ylim(0, 1.1)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(['0', '0.5', '1.0'])
    
    color_line = "#6A51A3" if not is_eta2 else "#2171B5"
    color_fill_hdi = "#9E9AC8" if not is_eta2 else "#6BAED6"
    color_fill_out = "#DADAEB" if not is_eta2 else "#BDD7E7"
    
    ax.plot(x, y, color=color_line, linewidth=4, zorder=3)
    ax.fill_between(x, 0, y, where=(x >= hdi_low) & (x <= hdi_high), color=color_fill_hdi, alpha=0.7, zorder=2, interpolate=True)
    ax.fill_between(x, 0, y, where=(x < hdi_low), color=color_fill_out, alpha=0.4, zorder=2, interpolate=True)
    ax.fill_between(x, 0, y, where=(x > hdi_high), color=color_fill_out, alpha=0.4, zorder=2, interpolate=True)
    
    ax.axvline(0, color='black', linestyle='--', linewidth=2.5, zorder=1, alpha=0.8)
    if real_interaction_val is not None:
        ax.axvline(real_interaction_val, color='#E41A1C', linestyle='-', linewidth=4, zorder=4)
        
    if format_pct:
        stats_text = f"Mean = {mean_val:.2f}%\n95% HDI = [{hdi_low:.2f}%, {hdi_high:.2f}%]\nP(>0) = {p_val*100:.1f}%"
        x_label = "Interaction Effect Size (Accuracy Difference %)"
    elif is_eta2:
        stats_text = f"Mean = {mean_val:.3f}\n95% HDI = [{hdi_low:.3f}, {hdi_high:.3f}]\nP(>0) = {p_val*100:.1f}%"
        x_label = r"Interaction Effect Size (Partial $\eta^2$)"
    else:
        stats_text = f"Mean = {mean_val:.3f}\n95% HDI = [{hdi_low:.3f}, {hdi_high:.3f}]\nP(>0) = {p_val*100:.1f}%"
        x_label = "Interaction Effect Size (Reaction Time Difference in seconds)"

    ax.text(1.0, 1.05, stats_text, transform=ax.transAxes, fontsize=20, fontweight='bold', va='bottom', ha='right', zorder=5)

    handles = [mpatches.Patch(color=color_fill_hdi, alpha=0.7, label='95% HDI')]
    if real_interaction_val is not None:
        handles.append(mlines.Line2D([], [], color='#E41A1C', lw=4, label='Observed Data'))
        
    ax.legend(
        handles=handles, loc='lower left', bbox_to_anchor=(0.0, 1.05), 
        ncol=1, frameon=False, prop={'size': 18, 'weight': 'bold'}
    )

    ax.set_xlabel(x_label, fontsize=24, fontweight='bold', color='black', labelpad=15)
    ax.set_ylabel("Density", fontsize=28, fontweight='bold', color='black', labelpad=15) 
    
    plt.tight_layout()
    output_dir.mkdir(parents=True, exist_ok=True)
    save_path = output_dir / f"PPC_{effect_name.title()}.png"
    
    plt.savefig(save_path, dpi=400, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"[SUCCESS] Publication interaction plot saved to: {save_path}")

In [ ]:
import gc
from pathlib import Path

# (Assuming libraries like hddm, arviz as az, matplotlib.pyplot as plt 
# and constants like MODELS_TO_RUN, BASE_OUTPUT_DIR are in the global namespace)

# ==============================================================================
# Module 5: Batch Pipeline Execution & Memory Management
# Description: Orchestrates the sequential loading of models, execution of 
#              Posterior Predictive Checks (PPC), statistical inference, 
#              generation of publication plots, and aggressive memory garbage 
#              collection to prevent out-of-memory (OOM) errors during batching.
# ==============================================================================

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*70}")
    print(f"[INFO] Initiating Batch Processing Pipeline for Model: {model_name}")
    print(f"{'='*70}")
    
    # Ensure each model has an independent, dedicated output directory
    model_out_dir = Path(BASE_OUTPUT_DIR) / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)
    
    # Construct the path for the serialized .nc model file
    nc_path = Path(BASE_INPUT_DIR) / model_name / f"{model_name}_fitted.nc"
    
    if not nc_path.exists():
        print(f"[WARNING] Model file not located at: {nc_path}. Skipping to next model.")
        continue

    # --------------------------------------------------------------------------
    # Phase 1: Load Serialized Model Data
    # --------------------------------------------------------------------------
    print("[INFO] Loading NetCDF posterior traces and HDDM model structure...")
    idata = az.from_netcdf(nc_path)
    # Cast Path object to string explicitly for compatibility with HDDM's load method
    model = hddm.load(str(nc_path)) 
    
    # --------------------------------------------------------------------------
    # Phase 2: Generate or Retrieve Posterior Predictive Checks (PPC)
    # --------------------------------------------------------------------------
    ppc_data = get_or_generate_ppc_data(
        model=model, 
        idata=idata, 
        condition_col=CONDITION_COL, 
        samples=PPC_SAMPLES, 
        model_name=model_name
    )
    
    # --------------------------------------------------------------------------
    # Phase 3: Baseline Statistical Computation and Visualization
    # --------------------------------------------------------------------------
    print("[INFO] Executing baseline statistics and generating collapsed condition plots...")
    
    # Generate Bayesian p-value and High-Density Interval (HDI) summary table
    stats_table = save_ppc_stats_table(
        model, ppc_data, CONDITION_COL, model_name=model_name
    )
    
    # Render publication-ready 2xN condition-level density plots
    plot_hddm_ppc_collapsed_conditions(
        model, ppc_data, CONDITION_COL, model_name=model_name
    )
    
    # --------------------------------------------------------------------------
    # Phase 4: Compute Effect-Level PPC Distributions
    # --------------------------------------------------------------------------
    print("[INFO] Computing posterior predictive distributions for target effects (Main & Interaction)...")
    effects_df, summary_df = compute_ppc_effects(
        model=model, 
        ppc_data=ppc_data, 
        condition_cols=['judgment_condition', 'pair_type']
    )
    
    effects_save_path = model_out_dir / "PPC_Effects_Samples.csv"
    summary_save_path = model_out_dir / "PPC_Effects_HDI_Summary.csv"
    
    effects_df.to_csv(effects_save_path, index=False, encoding='utf-8-sig')
    summary_df.to_csv(summary_save_path, index=False, encoding='utf-8-sig')
    print(f"[SUCCESS] Posterior predictive effect data exported to: {model_out_dir}")

    # --------------------------------------------------------------------------
    # Phase 5: Advanced Visualization (PPC vs. Empirical Effects)
    # --------------------------------------------------------------------------
    print("[INFO] Rendering comparative visual suite (Predictive vs. Empirical Effects)...")
    
    # Calculate effect sizes directly from the empirical dataset (Raw & Partial Eta-Squared)
    real_data_effects = compute_real_effects(model.data)
    
    # Generate 3x3 panoramic matrices comparing all effect distributions for Accuracy and RT
    plot_ppc_effects(
        samples_df=effects_df, summary_df=summary_df, 
        real_effects=real_data_effects, metric='acc', output_dir=model_out_dir
    )
    plot_ppc_effects(
        samples_df=effects_df, summary_df=summary_df, 
        real_effects=real_data_effects, metric='rt', output_dir=model_out_dir
    )

    # Generate publication-grade singular interaction plots
    # -> ACC Interaction (Raw Percentage Difference)
    plot_publication_interaction(
        effects_save_path, summary_save_path, 
        real_interaction_val=real_data_effects.get('acc_interaction'), 
        metric='acc', is_eta2=False, output_dir=model_out_dir
    )
    # -> ACC Interaction (Partial Eta Squared)
    plot_publication_interaction(
        effects_save_path, summary_save_path, 
        real_interaction_val=real_data_effects.get('acc_interaction_eta2'), 
        metric='acc', is_eta2=True, output_dir=model_out_dir
    )
    # -> RT Interaction (Raw Difference in Seconds)
    plot_publication_interaction(
        effects_save_path, summary_save_path, 
        real_interaction_val=real_data_effects.get('rt_interaction'), 
        metric='rt', is_eta2=False, output_dir=model_out_dir
    )
    # -> RT Interaction (Partial Eta Squared)
    plot_publication_interaction(
        effects_save_path, summary_save_path, 
        real_interaction_val=real_data_effects.get('rt_interaction_eta2'), 
        metric='rt', is_eta2=True, output_dir=model_out_dir
    )

    # --------------------------------------------------------------------------
    # Phase 6: Export Aligned Trial-Level Data
    # --------------------------------------------------------------------------
    print("[INFO] Exporting aligned trial-level PPC simulations for downstream Frequentist analyses...")
    
    csv_single_path = model_out_dir / "Master_PPC_Simulations_Single.csv"
    df_single_ppc = export_aligned_ppc_data(
        model=model, ppc_df=ppc_data,
        save_path=csv_single_path, export_mode="single_random"
    )
    
    csv_agg_path = model_out_dir / "Master_PPC_Simulations_Aggregated.csv"
    df_agg_ppc = export_aligned_ppc_data(
        model=model, ppc_df=ppc_data,
        save_path=csv_agg_path, export_mode="aggregate"
    )
    
    print(f"[SUCCESS] All analytical procedures for model '{model_name}' completed.")

    # --------------------------------------------------------------------------
    # Phase 7: Aggressive Memory Cleanup
    # --------------------------------------------------------------------------
    print(f"[INFO] Freeing memory allocations for {model_name}...")
    
    # Explicitly delete large data structures
    del idata
    del model
    del ppc_data
    del stats_table
    del df_single_ppc
    del df_agg_ppc
    del effects_df
    del summary_df
    del real_data_effects
    
    # Close all active matplotlib figures to release RAM
    plt.close('all') 
    
    # Force Python's Garbage Collector to reclaim unreferenced memory
    gc.collect()
    
    print("[SUCCESS] Memory cleanup successful.\n")

print("======================================================================")
print("[SUCCESS] PIPELINE COMPLETE: All specified models have been processed.")
print("======================================================================")